In [ ]:
import requests
import json
import polars as pl
import pyspark.sql as ps
from pyspark.sql import SparkSession
import os
from dotenv import load_dotenv
from datetime import datetime
import logging

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

logger = logging.getLogger(__name__)

Estructura de alamacen en DataBricks
/dbfs/FileStore/bronze/weather/
    city=rosario/
        date=2026-03-20/
            2026-03-20T06-00-00Z.json
            2026-03-20T12-00-00Z.json
    city=rafaela/
        date=2026-03-20/


guardar el dato en la hora
datetime.utcnow()

In [10]:
load_dotenv()

api_key = os.getenv('clave_api_clima')

base_url = "http://api.weatherapi.com/v1"

ciudades = [
    "Venado Tuerto", "Rosario", "Firmat", "Rafaela",
    "Casilda", "Cañada de Gómez", "San Lorenzo",
    "El Trebol", "San Justo"
]

# timestamp
now = datetime.utcnow()
date_str = now.strftime("%Y-%m-%d")
timestamp = now.strftime("%Y-%m-%dT%H-%M-%SZ")

# endpoints dinámicos
endpoints = {
    "forecast": lambda ciudad: f"{base_url}/forecast.json?key={api_key}&q={ciudad}&days=3&aqi=yes&alerts=yes",
    "history": lambda ciudad: f"{base_url}/history.json?key={api_key}&q={ciudad}&dt={date_str}",
    "astronomy": lambda ciudad: f"{base_url}/astronomy.json?key={api_key}&q={ciudad}"
}

path_raiz = "/dbfs/FileStore/tables/bronce/clima"

for ciudad in ciudades:
    city_folder = ciudad.lower().replace(" ", "_")

    for endpoint_name, url_func in endpoints.items():
        url = url_func(ciudad)

        try:
            response = requests.get(url)

            if response.status_code == 200:
                raw_data = response.json()

                full_folder_path = f"{path_raiz}/{endpoint_name}/city={city_folder}/date={date_str}"
                os.makedirs(full_folder_path, exist_ok=True)

                file_name = f"{timestamp}.json"
                full_file_path = f"{full_folder_path}/{file_name}"

                with open(full_file_path, "w", encoding="utf-8") as f:
                    json.dump(raw_data, f, ensure_ascii=False)

                logging.info(f"Guardado OK: {endpoint_name} | {city_folder}")

            else:
                logging.error(f"Error {response.status_code} en {endpoint_name} - {ciudad}")

        except Exception as e:
            logging.exception(f"Fallo crítico en {endpoint_name} - {ciudad}")


2026-03-20 16:24:07,831 - INFO - Guardado OK: forecast | venado_tuerto
2026-03-20 16:24:08,151 - INFO - Guardado OK: history | venado_tuerto
2026-03-20 16:24:08,449 - INFO - Guardado OK: astronomy | venado_tuerto
2026-03-20 16:24:08,755 - INFO - Guardado OK: forecast | rosario
2026-03-20 16:24:10,061 - INFO - Guardado OK: history | rosario
2026-03-20 16:24:10,392 - INFO - Guardado OK: astronomy | rosario
2026-03-20 16:24:10,691 - INFO - Guardado OK: forecast | firmat
2026-03-20 16:24:10,994 - INFO - Guardado OK: history | firmat
2026-03-20 16:24:11,290 - INFO - Guardado OK: astronomy | firmat
2026-03-20 16:24:11,620 - INFO - Guardado OK: forecast | rafaela
2026-03-20 16:24:11,961 - INFO - Guardado OK: history | rafaela
2026-03-20 16:24:12,261 - INFO - Guardado OK: astronomy | rafaela
2026-03-20 16:24:12,584 - INFO - Guardado OK: forecast | casilda
2026-03-20 16:24:12,861 - INFO - Guardado OK: history | casilda
2026-03-20 16:24:13,161 - INFO - Guardado OK: astronomy | casilda
2026-03-20